## SVD — quick cross-check (my own, for validation)

A simple Surprise SVD with default params, evaluated on the SAME candidate pools
as TwoTower/Popularity. NOT the final SVD (Kate's tuned model is the official one) —
this is for pipeline validation and a quick neural-vs-classical comparison.

In [2]:
import numpy as np
import pandas as pd
from sklearn.metrics import ndcg_score
from surprise import SVD, Dataset, Reader

RANDOM_STATE = 42

In [3]:
# Load Data

train = pd.read_csv("train_ratings.csv")
test  = pd.read_csv("test_ratings.csv")

train_items_by_user = train.groupby("user_id")["book_id"].apply(set).to_dict()
test_ratings_by_user = (
    test.groupby("user_id")
        .apply(lambda g: dict(zip(g["book_id"], g["rating"])))
        .to_dict()
)
all_items = train["book_id"].unique()
book_popularity = train["book_id"].value_counts().to_dict()

print(f"Users with test data: {len(test_ratings_by_user):,}")

Users with test data: 58,787


In [4]:
def build_candidate_pools(test_ratings_by_user, train_items_by_user,
                          all_items, n_neg=100, seed=RANDOM_STATE):
    rng = np.random.default_rng(seed)
    all_items_arr = np.asarray(all_items)
    pools = {}
    for user, test_items in test_ratings_by_user.items():
        seen = train_items_by_user.get(user, set())
        pos_items = set(test_items.keys())
        exclude = seen | pos_items
        negs = []
        while len(negs) < n_neg:
            cand = rng.choice(all_items_arr, size=n_neg * 2, replace=False)
            negs = [it for it in cand if it not in exclude][:n_neg]
        pools[user] = list(pos_items) + negs
    return pools

candidate_pools = build_candidate_pools(
    test_ratings_by_user, train_items_by_user, all_items, n_neg=100
)
print(f"Built candidate pools for {len(candidate_pools):,} users")

Built candidate pools for 58,787 users


In [5]:
def evaluate_ndcg(score_fn, candidate_pools, test_ratings_by_user, k=10):
    ndcgs = []
    for user, items in candidate_pools.items():
        ratings = test_ratings_by_user[user]
        y_true = [ratings.get(it, 0) for it in items]
        if sum(y_true) == 0 or len(set(y_true)) == 1:
            continue
        y_score = score_fn(user, items)
        ndcgs.append(ndcg_score([y_true], [y_score], k=k))
    return float(np.mean(ndcgs)), len(ndcgs)

In [6]:
# Train SVD

reader = Reader(rating_scale=(train["rating"].min(), train["rating"].max()))
surprise_data = Dataset.load_from_df(train[["user_id", "book_id", "rating"]], reader)
trainset = surprise_data.build_full_trainset()

svd_model = SVD(random_state=RANDOM_STATE)   # default params
svd_model.fit(trainset)
print("SVD trained")

SVD trained


In [13]:
# Confirm we can access SVD's latent factors and id mapping
print("pu (user factors) shape:", svd_model.pu.shape)
print("qi (item factors) shape:", svd_model.qi.shape)
print("trainset available:", 'trainset' in dir())
# test id conversion on one user
u0 = next(iter(candidate_pools))
try:
    inner_u = trainset.to_inner_uid(u0)
    print(f"user {u0[:8]} -> inner id {inner_u} -> factor vector shape {svd_model.pu[inner_u].shape}")
except Exception as e:
    print(f"id conversion issue: {e}")

pu (user factors) shape: (61078, 100)
qi (item factors) shape: (22931, 100)
trainset available: True
user 00004584 -> inner id 14170 -> factor vector shape (100,)


In [14]:
def svd_ranking_score_fn(user, items):
    """SVD as a ranking model: score = user latent factor . item latent factor.
    Uses raw pu/qi dot product (no biases, no global mean) — the closest analogue
    to TwoTower's latent-vector similarity. Lets us compare MF vs neural on equal
    footing (both rank by latent-vector similarity), isolating architecture."""
    inner_u = trainset.to_inner_uid(user)
    u_vec = svd_model.pu[inner_u]                      # (100,)
    scores = []
    for it in items:
        try:
            inner_i = trainset.to_inner_iid(it)
            scores.append(float(np.dot(u_vec, svd_model.qi[inner_i])))
        except ValueError:
            scores.append(-np.inf)                     # item unseen in train → lowest
    return scores

svd_rank_ndcg, n_users = evaluate_ndcg(
    svd_ranking_score_fn, candidate_pools, test_ratings_by_user, k=10
)
print(f"SVD (ranking, dot product) — graded NDCG@10: {svd_rank_ndcg:.4f}  (over {n_users:,} users)")
print(f"Compare: SVD-as-rating(est) 0.1587 | Popularity 0.6992 | TwoTower 0.8508")

SVD (ranking, dot product) — graded NDCG@10: 0.1909  (over 58,787 users)
Compare: SVD-as-rating(est) 0.1587 | Popularity 0.6992 | TwoTower 0.8508


In [11]:
from surprise import accuracy

# RMSE is evaluated on the test set's actual ratings (rating-prediction task).
# No candidate pools / negatives — just predicted rating vs true rating.
test_predictions = [
    svd_model.predict(row.user_id, row.book_id, r_ui=row.rating)
    for row in test.itertuples(index=False)
]

rmse = accuracy.rmse(test_predictions, verbose=False)
mae  = accuracy.mae(test_predictions, verbose=False)
print(f"SVD (default) — test RMSE: {rmse:.4f}")
print(f"SVD (default) — test MAE:  {mae:.4f}")
print(f"(Kate's SVD: RMSE 0.7614 default / 0.7627 tuned)")

SVD (default) — test RMSE: 0.7614
SVD (default) — test MAE:  0.5848
(Kate's SVD: RMSE 0.7614 default / 0.7627 tuned)


In [7]:
def svd_score_fn(user, items):
    return [svd_model.predict(user, it).est for it in items]

svd_ndcg, n_users = evaluate_ndcg(svd_score_fn, candidate_pools, test_ratings_by_user, k=10)
print(f"SVD (default) — graded NDCG@10: {svd_ndcg:.4f}  (over {n_users:,} users)")
print(f"Reference: Popularity 0.6992 | TwoTower 0.8508 (same pools)")

SVD (default) — graded NDCG@10: 0.1587  (over 58,787 users)
Reference: Popularity 0.6992 | TwoTower 0.8508 (same pools)


In [8]:
# Diagnostic: check what SVD actually predicts for one user's pool
u0 = next(iter(candidate_pools))
items0 = candidate_pools[u0]

preds = [svd_model.predict(u0, it) for it in items0[:10]]
for p in preds:
    print(f"item={p.iid}  est={p.est:.4f}  impossible={p.details.get('was_impossible')}")

item=6250208  est=3.5891  impossible=False
item=1258121  est=3.4978  impossible=False
item=297676  est=3.7356  impossible=False
item=3636  est=4.3570  impossible=False
item=11788115  est=3.5422  impossible=False
item=14823888  est=3.5283  impossible=False
item=982432  est=4.0321  impossible=False
item=20307024  est=4.1116  impossible=False
item=23600172  est=3.7238  impossible=False
item=92637  est=3.6021  impossible=False


In [9]:
# Diagnostic: are SVD's scores for POSITIVES actually higher than for NEGATIVES?
u0 = next(iter(candidate_pools))
items0 = candidate_pools[u0]
ratings0 = test_ratings_by_user[u0]

pos_est, neg_est = [], []
for it in items0:
    est = svd_model.predict(u0, it).est
    if it in ratings0:
        pos_est.append(est)
    else:
        neg_est.append(est)

print(f"User {u0[:8]}")
print(f"  positives: {len(pos_est)} items, est mean={np.mean(pos_est):.3f}, range=[{min(pos_est):.2f}, {max(pos_est):.2f}]")
print(f"  negatives: {len(neg_est)} items, est mean={np.mean(neg_est):.3f}, range=[{min(neg_est):.2f}, {max(neg_est):.2f}]")
print(f"  → positives scored higher? {np.mean(pos_est) > np.mean(neg_est)}")

User 00004584
  positives: 4 items, est mean=3.795, range=[3.50, 4.36]
  negatives: 100 items, est mean=3.784, range=[3.06, 4.60]
  → positives scored higher? True


In [10]:
# Is the positive/negative est gap small across MANY users, not just one?
gaps = []
for user, items in list(candidate_pools.items())[:2000]:   # 2000 users
    ratings = test_ratings_by_user[user]
    pos, neg = [], []
    for it in items:
        est = svd_model.predict(user, it).est
        (pos if it in ratings else neg).append(est)
    if pos and neg:
        gaps.append(np.mean(pos) - np.mean(neg))

gaps = np.array(gaps)
print(f"Over {len(gaps)} users:")
print(f"  mean (pos_est - neg_est): {gaps.mean():.4f}")
print(f"  fraction where positives scored higher: {(gaps > 0).mean():.1%}")

Over 2000 users:
  mean (pos_est - neg_est): 0.1083
  fraction where positives scored higher: 75.7%
